In [ ]:
import pandas as pd
from tqdm import tqdm

eval_df = pd.read_csv("evaluation_queries.csv")

In [ ]:
def recall_at_k(results, relevant_id, k):
    top_k_ids = [r["id"] for r in results[:k]]
    return int(relevant_id in top_k_ids)

In [ ]:
def reciprocal_rank(results, relevant_id):
    for rank, r in enumerate(results, start=1):
        if r["id"] == relevant_id:
            return 1 / rank
    return 0

In [ ]:
def evaluate_retriever(retriever, name, k_values=[5, 10]):
    
    recall_scores = {k: [] for k in k_values}
    mrr_scores = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        query = row["query"]
        relevant_id = row["relevant_doc_id"]

        results = retriever.search(query, top_k=10)

        # Recall@K
        for k in k_values:
            recall_scores[k].append(
                recall_at_k(results, relevant_id, k)
            )

        # MRR
        mrr_scores.append(
            reciprocal_rank(results, relevant_id)
        )

    # Compute averages
    recall_avg = {k: sum(v)/len(v) for k, v in recall_scores.items()}
    mrr_avg = sum(mrr_scores) / len(mrr_scores)

    print(f"\n===== {name} =====")
    for k in k_values:
        print(f"Recall@{k}: {recall_avg[k]:.4f}")
    print(f"MRR: {mrr_avg:.4f}")

    return recall_avg, mrr_avg

In [ ]:
bm25_metrics = evaluate_retriever(bm25_retriever, "BM25")
dense_metrics = evaluate_retriever(dense_retriever, "Dense")
hybrid_metrics = evaluate_retriever(hybrid_retriever, "Hybrid (RRF)")